# GroupX Full Training: SelectiveCL Reproduction

This notebook reproduces the AGD20K-Seen setting for **Selective Contrastive Learning for Weakly Supervised Affordance Grounding (ICCV 2025)**.

The notebook is written for the SKKU GPU server. Heavy full training is disabled by default because the completed full run already exists in `full_runs/20260508_023621`.

## Reproduction Goal

Weakly supervised affordance grounding (WSAG) localizes the object region that supports a given action. SelectiveCL learns this localization from image-level action labels and third-person exocentric examples.

- Main target: AGD20K-Seen reproduction.
- Main outputs: training setup, training log summary, official checkpoint evaluation, and metric comparison.
- Main metrics: KLD, SIM, and NSS.
- Dense ground-truth heatmaps are available for the test split, so grounding metrics are reported on the test split. Training progress is summarized with training losses and classification accuracy logs.

## Sources

- Official ICCV paper: https://openaccess.thecvf.com/content/ICCV2025/html/Moon_Selective_Contrastive_Learning_for_Weakly_Supervised_Affordance_Grounding_ICCV_2025_paper.html
- Supplemental material: https://openaccess.thecvf.com/content/ICCV2025/supplemental/Moon_Selective_Contrastive_Learning_ICCV_2025_supplemental.pdf
- Official code: https://github.com/hynnsk/SelectiveCL

In [1]:
#OWN CODE: Define paths and constants used throughout the reproduction notebook.
from pathlib import Path
import os
import re
import subprocess
import sys

ROOT = Path('/root/workspace/andycho/CV/SelectiveCL')
DATA_ROOT = Path('/root/workspace/andycho/CV/AGD20K')
FULL_RUN_DIR = ROOT / 'full_runs' / '20260508_023621'
SEEN_CHECKPOINT = ROOT / 'checkpoints' / 'agd20k_seen.pth'
UNSEEN_CHECKPOINT = ROOT / 'checkpoints' / 'agd20k_unseen.pth'

os.chdir(ROOT)
print('Working directory:', Path.cwd())

Working directory: /root/workspace/andycho/CV/SelectiveCL


In [2]:
#OWN CODE: Verify the server environment and required local assets.
def show_exists(label, path):
    path = Path(path)
    print(f'{label}: {path} -> {path.exists()}')

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

try:
    import torch
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA device count:', torch.cuda.device_count())
except Exception as exc:
    print('Torch import failed:', repr(exc))

show_exists('SelectiveCL repository', ROOT)
show_exists('AGD20K data root', DATA_ROOT)
show_exists('Completed full run directory', FULL_RUN_DIR)
show_exists('AGD20K-Seen checkpoint', SEEN_CHECKPOINT)
show_exists('AGD20K-Unseen checkpoint', UNSEEN_CHECKPOINT)

Python executable: /root/anaconda3/envs/selectivecl/bin/python
Python version: 3.7.16


Torch version: 1.9.0
CUDA available: True
CUDA device count: 4
SelectiveCL repository: /root/workspace/andycho/CV/SelectiveCL -> True
AGD20K data root: /root/workspace/andycho/CV/AGD20K -> True
Completed full run directory: /root/workspace/andycho/CV/SelectiveCL/full_runs/20260508_023621 -> True
AGD20K-Seen checkpoint: /root/workspace/andycho/CV/SelectiveCL/checkpoints/agd20k_seen.pth -> True
AGD20K-Unseen checkpoint: /root/workspace/andycho/CV/SelectiveCL/checkpoints/agd20k_unseen.pth -> True


/root/anaconda3/envs/selectivecl/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Setup

The AGD20K directory follows the LOCATE-style layout used by the official SelectiveCL repository.

- `Seen/trainset/exocentric`: third-person interaction examples.
- `Seen/trainset/egocentric`: object-focused target images.
- `Seen/testset/egocentric`: test images.
- `Seen/testset/GT`: dense affordance heatmaps for evaluation.

In [3]:
#OWN CODE: Count dataset files for the AGD20K-Seen and AGD20K-Unseen splits.
def count_files(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for item in path.rglob('*') if item.is_file())

dataset_rows = [
    {'Split': 'Seen', 'Subset': 'train exocentric', 'Count': count_files(DATA_ROOT / 'Seen/trainset/exocentric')},
    {'Split': 'Seen', 'Subset': 'train egocentric', 'Count': count_files(DATA_ROOT / 'Seen/trainset/egocentric')},
    {'Split': 'Seen', 'Subset': 'test egocentric', 'Count': count_files(DATA_ROOT / 'Seen/testset/egocentric')},
    {'Split': 'Unseen', 'Subset': 'train exocentric', 'Count': count_files(DATA_ROOT / 'Unseen/trainset/exocentric')},
    {'Split': 'Unseen', 'Subset': 'train egocentric', 'Count': count_files(DATA_ROOT / 'Unseen/trainset/egocentric')},
    {'Split': 'Unseen', 'Subset': 'test egocentric', 'Count': count_files(DATA_ROOT / 'Unseen/testset/egocentric')},
]

try:
    import pandas as pd
    display(pd.DataFrame(dataset_rows))
except ImportError:
    for row in dataset_rows:
        print(row)

{'Split': 'Seen', 'Subset': 'train exocentric', 'Count': 20061}
{'Split': 'Seen', 'Subset': 'train egocentric', 'Count': 6929}
{'Split': 'Seen', 'Subset': 'test egocentric', 'Count': 1710}
{'Split': 'Unseen', 'Subset': 'train exocentric', 'Count': 13323}
{'Split': 'Unseen', 'Subset': 'train egocentric', 'Count': 4815}
{'Split': 'Unseen', 'Subset': 'test egocentric', 'Count': 1080}


## Model Pipeline

SelectiveCL combines dense visual features, text-conditioned object discovery, and contrastive learning.

1. DINO ViT-S/16 extracts dense visual features.
2. CLIP ViT-B/16 creates object affinity maps from action prompts.
3. A shared classifier produces class activation maps for egocentric and exocentric views.
4. Selective prototypical contrastive learning uses part-level prototypes when they are reliable and object-level prototypes otherwise.
5. Pixel contrastive learning separates affordance-relevant and affordance-irrelevant pixels.
6. Inference calibrates CAM output with the CLIP object affinity map.

In [4]:
#OWN CODE: Summarize the paper-matching hyperparameters used in the completed full run.
hyperparameter_rows = [
    {'Item': 'Backbone', 'Paper': 'DINO ViT-S/16 + CLIP ViT-B/16', 'Reproduction': 'DINO ViT-S/16 + CLIP ViT-B/16'},
    {'Item': 'Dataset', 'Paper': 'AGD20K-Seen', 'Reproduction': 'AGD20K-Seen'},
    {'Item': 'Epochs', 'Paper': '15', 'Reproduction': '15'},
    {'Item': 'Batch size', 'Paper': '8', 'Reproduction': '8'},
    {'Item': 'Learning rate', 'Paper': '1e-3', 'Reproduction': '1e-3'},
    {'Item': 'Weight decay', 'Paper': '5e-4', 'Reproduction': '5e-4'},
    {'Item': 'Exocentric images E', 'Paper': '3', 'Reproduction': '3'},
    {'Item': 'alpha and gamma', 'Paper': '0.6', 'Reproduction': '0.6'},
    {'Item': 'Temperature tau', 'Paper': '0.5', 'Reproduction': '0.5'},
]

try:
    import pandas as pd
    display(pd.DataFrame(hyperparameter_rows))
except ImportError:
    for row in hyperparameter_rows:
        print(row)

{'Item': 'Backbone', 'Paper': 'DINO ViT-S/16 + CLIP ViT-B/16', 'Reproduction': 'DINO ViT-S/16 + CLIP ViT-B/16'}
{'Item': 'Dataset', 'Paper': 'AGD20K-Seen', 'Reproduction': 'AGD20K-Seen'}
{'Item': 'Epochs', 'Paper': '15', 'Reproduction': '15'}
{'Item': 'Batch size', 'Paper': '8', 'Reproduction': '8'}
{'Item': 'Learning rate', 'Paper': '1e-3', 'Reproduction': '1e-3'}
{'Item': 'Weight decay', 'Paper': '5e-4', 'Reproduction': '5e-4'}
{'Item': 'Exocentric images E', 'Paper': '3', 'Reproduction': '3'}
{'Item': 'alpha and gamma', 'Paper': '0.6', 'Reproduction': '0.6'}
{'Item': 'Temperature tau', 'Paper': '0.5', 'Reproduction': '0.5'}


## Completed Full Training Run

The completed full run is stored in `full_runs/20260508_023621`.

- Seen training started at 2026-05-08 02:36:35 and finished at 2026-05-08 07:46:06.
- Unseen training started at 2026-05-08 07:46:06 and finished at 2026-05-08 11:01:29.
- Official Seen checkpoint evaluation finished at 2026-05-08 11:03:42.
- Official Unseen checkpoint evaluation finished at 2026-05-08 11:04:31.

In [5]:
#OWN CODE: Parse epoch-level metric summaries from the SelectiveCL training logs.
epoch_pattern = re.compile(r'epoch\|mKLD\|mSIM\|mNSS , (\d+), ([\d.]+), ([\d.]+), ([\-\d.]+)')
refined_pattern = re.compile(
    r'refined ego-ego \+ mKLD\|mSIM\|mNSS = ([\d.]+), ([\d.]+), ([\-\d.]+), '
    r'BEST e\|mKLD\|mSIM\|mNSS , (\d+), ([\d.]+), ([\d.]+), ([\-\d.]+)'
)

def parse_training_log(path, split):
    rows = []
    last_epoch = None
    for line in Path(path).read_text().splitlines():
        epoch_match = epoch_pattern.search(line)
        if epoch_match:
            last_epoch = int(epoch_match.group(1))
            rows.append({
                'Split': split,
                'Epoch': last_epoch,
                'Prediction': 'ego_pred',
                'KLD': float(epoch_match.group(2)),
                'SIM': float(epoch_match.group(3)),
                'NSS': float(epoch_match.group(4)),
            })
            continue

        refined_match = refined_pattern.search(line)
        if refined_match and last_epoch is not None:
            rows.append({
                'Split': split,
                'Epoch': last_epoch,
                'Prediction': 'refined ego-ego',
                'KLD': float(refined_match.group(1)),
                'SIM': float(refined_match.group(2)),
                'NSS': float(refined_match.group(3)),
            })
    return rows

training_rows = []
training_rows += parse_training_log(FULL_RUN_DIR / 'train_seen.log', 'Seen')
training_rows += parse_training_log(FULL_RUN_DIR / 'train_unseen.log', 'Unseen')

try:
    import pandas as pd
    training_df = pd.DataFrame(training_rows)
    display(training_df[training_df['Split'].eq('Seen')].tail(10))
except ImportError:
    for row in training_rows[-20:]:
        print(row)

{'Split': 'Unseen', 'Epoch': 6, 'Prediction': 'ego_pred', 'KLD': 1.276, 'SIM': 0.388, 'NSS': 1.346}
{'Split': 'Unseen', 'Epoch': 6, 'Prediction': 'refined ego-ego', 'KLD': 1.242, 'SIM': 0.41, 'NSS': 1.338}
{'Split': 'Unseen', 'Epoch': 7, 'Prediction': 'ego_pred', 'KLD': 1.307, 'SIM': 0.38, 'NSS': 1.31}
{'Split': 'Unseen', 'Epoch': 7, 'Prediction': 'refined ego-ego', 'KLD': 1.274, 'SIM': 0.403, 'NSS': 1.308}
{'Split': 'Unseen', 'Epoch': 8, 'Prediction': 'ego_pred', 'KLD': 1.302, 'SIM': 0.376, 'NSS': 1.338}
{'Split': 'Unseen', 'Epoch': 8, 'Prediction': 'refined ego-ego', 'KLD': 1.258, 'SIM': 0.403, 'NSS': 1.332}
{'Split': 'Unseen', 'Epoch': 9, 'Prediction': 'ego_pred', 'KLD': 1.333, 'SIM': 0.372, 'NSS': 1.278}
{'Split': 'Unseen', 'Epoch': 9, 'Prediction': 'refined ego-ego', 'KLD': 1.287, 'SIM': 0.399, 'NSS': 1.289}
{'Split': 'Unseen', 'Epoch': 10, 'Prediction': 'ego_pred', 'KLD': 1.286, 'SIM': 0.387, 'NSS': 1.343}
{'Split': 'Unseen', 'Epoch': 10, 'Prediction': 'refined ego-ego', 'KLD': 1

In [6]:
#OWN CODE: Extract the best local full-training result for AGD20K-Seen.
seen_refined_rows = [
    row for row in training_rows
    if row['Split'] == 'Seen' and row['Prediction'] == 'refined ego-ego'
]
best_seen = min(seen_refined_rows, key=lambda row: row['KLD'])
print('Best local full-training result on AGD20K-Seen:')
print(best_seen)

Best local full-training result on AGD20K-Seen:
{'Split': 'Seen', 'Epoch': 4, 'Prediction': 'refined ego-ego', 'KLD': 1.136, 'SIM': 0.427, 'NSS': 1.278}


In [7]:
#OWN CODE: Parse official checkpoint evaluation logs.
test_metric_pattern = re.compile(r'^(KLD|reeKLD|remKLD), (SIM|reeSIM|remSIM), (NSS|reeNSS|remNSS), ([\d.]+), ([\d.]+), ([\-\d.]+)$')

def parse_test_log(path, split):
    rows = []
    prediction_names = {'KLD': 'ego_pred', 'reeKLD': 'refined ego-ego', 'remKLD': 'refined ego-mean'}
    for line in Path(path).read_text().splitlines():
        match = test_metric_pattern.search(line.strip())
        if match:
            rows.append({
                'Split': split,
                'Prediction': prediction_names[match.group(1)],
                'KLD': float(match.group(4)),
                'SIM': float(match.group(5)),
                'NSS': float(match.group(6)),
            })
    return rows

official_rows = []
official_rows += parse_test_log(FULL_RUN_DIR / 'test_seen_official.log', 'Seen')
official_rows += parse_test_log(FULL_RUN_DIR / 'test_unseen_official.log', 'Unseen')

try:
    import pandas as pd
    display(pd.DataFrame(official_rows))
except ImportError:
    for row in official_rows:
        print(row)

{'Split': 'Seen', 'Prediction': 'ego_pred', 'KLD': 1.142, 'SIM': 0.415, 'NSS': 1.303}
{'Split': 'Seen', 'Prediction': 'refined ego-ego', 'KLD': 1.124, 'SIM': 0.433, 'NSS': 1.28}
{'Split': 'Seen', 'Prediction': 'refined ego-mean', 'KLD': 1.14, 'SIM': 0.422, 'NSS': 1.283}
{'Split': 'Unseen', 'Prediction': 'ego_pred', 'KLD': 1.287, 'SIM': 0.378, 'NSS': 1.377}
{'Split': 'Unseen', 'Prediction': 'refined ego-ego', 'KLD': 1.243, 'SIM': 0.405, 'NSS': 1.368}
{'Split': 'Unseen', 'Prediction': 'refined ego-mean', 'KLD': 1.245, 'SIM': 0.396, 'NSS': 1.378}


In [8]:
#OWN CODE: Compare paper, official checkpoint, and local full-training metrics.
comparison_rows = [
    {'Source': 'Paper Table 1', 'Prediction': 'Ours', 'KLD': 1.124, 'SIM': 0.433, 'NSS': 1.280},
    {'Source': 'Official checkpoint re-evaluation', 'Prediction': 'refined ego-ego', 'KLD': 1.124, 'SIM': 0.433, 'NSS': 1.280},
    {'Source': 'Local full training best', 'Prediction': 'refined ego-ego, epoch 4', 'KLD': 1.136, 'SIM': 0.427, 'NSS': 1.278},
]

try:
    import pandas as pd
    display(pd.DataFrame(comparison_rows))
except ImportError:
    for row in comparison_rows:
        print(row)

{'Source': 'Paper Table 1', 'Prediction': 'Ours', 'KLD': 1.124, 'SIM': 0.433, 'NSS': 1.28}
{'Source': 'Official checkpoint re-evaluation', 'Prediction': 'refined ego-ego', 'KLD': 1.124, 'SIM': 0.433, 'NSS': 1.28}
{'Source': 'Local full training best', 'Prediction': 'refined ego-ego, epoch 4', 'KLD': 1.136, 'SIM': 0.427, 'NSS': 1.278}


## Optional Full Training Command

The next cell is disabled by default. Enable it only when you intentionally want to rerun full AGD20K-Seen training. The completed run already provides the required reproduction evidence.

In [9]:
#OWN CODE: Keep the expensive full-training rerun disabled by default.
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    command = [
        sys.executable, 'train.py',
        '--data_root', str(DATA_ROOT),
        '--divide', 'Seen',
        '--exp_name', 'SelectiveCL_GroupX_Reproduction',
        '--gpu', '0',
        '--epochs', '15',
        '--batch_size', '8',
        '--lr', '0.001',
        '--weight_decay', '0.0005',
        '--alpha', '0.6',
        '--gamma1', '0.6',
        '--gamma2', '0.6',
        '--cont_temperature', '0.5',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Full training is disabled. Use the completed full run in full_runs/20260508_023621.')

Full training is disabled. Use the completed full run in full_runs/20260508_023621.


## Conclusion

The AGD20K-Seen reproduction follows the paper setup and produces a local full-training result close to the reported result. The official checkpoint re-evaluation exactly matches the paper's AGD20K-Seen metric: `KLD 1.124 / SIM 0.433 / NSS 1.280`.